## AlphaEarth vs LandTrendr — Side-by-Side Comparison (15 Sites)

**Purpose**: Compare AlphaEarth embedding-based and LandTrendr NDVI-based change detection
across 15 western U.S. study sites. Computes IoU (Intersection over Union) between both
algorithms' binary change masks.

**Inputs**:
- `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` (AlphaEarth, 2017–2024)
- 15 pre-computed LandTrendr GEE assets (see `src/config.py`)

**Outputs**: Side-by-side map layers; IoU metrics printed per site
**GEE auth required**: Yes

**Key parameters** (see `src/config.py`):
- AE threshold: 0.15 cosine dissimilarity
- LT threshold: 170 NDVI units
- Both use Gaussian smoothing radius=2, sigma=1 + morphological cleanup

**Expected runtime**: 5–20 min (EE computation per site, server-side)

**How to run**: Execute cells top-to-bottom. Change `SITE_INDEX` variable to switch sites.
See `src/config.py` for the full list of `LANDTRENDR_ASSETS` keys.


# AlphaEarth vs LandTrendr: Change Detection Comparison (15 Sites)

This notebook compares change detection results from two methods across 15 study sites:

- LandTrendr (precomputed assets in GEE)
- AlphaEarth embeddings (computed on-the-fly per AOI from GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL)

We standardize outputs into three layers for each method and combine all sites into single mosaicked layers for easy visual comparison:

- YOD/YOC: Year of change (end-year of lowest similarity or LandTrendr YOD)
- MAG: Magnitude of change
- DUR: Duration (LandTrendr duration vs. AlphaEarth count of pair-years above threshold)

Use the layer control to toggle between methods and explore differences by site.

In [ ]:
# ── Repo root on sys.path (works from repo root or notebooks/) ──────────────────
import sys
from pathlib import Path
_repo_root = Path.cwd()
if not (_repo_root / "src").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# Setup: imports and EE init
import ee, geemap
import math

from src.config import GEE_PROJECT

print("Authenticating/initializing Earth Engine…")
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    print("No active EE session found. Launching authentication flow…")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

# Utility: rectangle from center with span in degrees
# Note: Pure server-side (no getInfo), span is degrees (~0.4 as requested)
def rect_from_center(lon, lat, span_deg=0.4):
    half = ee.Number(span_deg).divide(2)
    return ee.Geometry.Rectangle([
        ee.Number(lon).subtract(half),
        ee.Number(lat).subtract(half),
        ee.Number(lon).add(half),
        ee.Number(lat).add(half)
    ], None, False)

# AOIs for 15 sites (0.4° squares)
aois = {
    # Texas Urbanization
    'Austin_Urban_TX': rect_from_center(-97.8, 30.3, 0.4),
    'Dallas_Urban_TX': rect_from_center(-96.8, 32.8, 0.4),
    'Houston_Urban_TX': rect_from_center(-95.37, 29.76, 0.4),

    # Oregon Urbanization
    'Portland_Urban_OR': rect_from_center(-122.7, 45.5, 0.4),
    'Bend_Urban_OR': rect_from_center(-121.3, 44.05, 0.4),

    # California Urbanization
    'Sacramento_Urban_CA': rect_from_center(-121.5, 38.6, 0.4),

    # Oregon Wildfires
    'Santiam_Fire_OR_2020': rect_from_center(-122.25, 44.75, 0.4),
    'Bootleg_Fire_OR_2021': rect_from_center(-121.25, 42.55, 0.4),

    # California Wildfires
    'Camp_Fire_CA_2018': rect_from_center(-121.6, 39.73, 0.4),
    'Dixie_Fire_CA_2021': rect_from_center(-121.3, 40.0, 0.4),
    'Mosquito_Fire_CA_2022': rect_from_center(-120.7, 39.1, 0.4),

    # Oregon Logging
    'CoosBay_Forest_OR': rect_from_center(-124.0, 43.35, 0.4),
    'MtHood_Forest_OR': rect_from_center(-121.8, 45.35, 0.4),

    # California Logging
    'ShastaTrinity_Forest_CA': rect_from_center(-122.3, 40.9, 0.4),

    # Texas Logging
    'Angelina_Forest_TX': rect_from_center(-94.75, 31.3, 0.4),
}

print(f"Defined {len(aois)} AOIs.")

In [ ]:
# ── Run-mode parameters ─────────────────────────────────────────────────────────
from src.config import AE_MAG_THRESHOLD, LT_MAG_THRESHOLD, AE_SMOOTH_RADIUS, AE_SMOOTH_SIGMA

# RUN_HEAVY = True  → process all 15 sites (slow, ~5–10 min)
# RUN_HEAVY = False → quick demo with QUICK_SITES only (~1–2 min)
RUN_HEAVY   = False
QUICK_SITES = ['Austin_Urban_TX', 'Camp_Fire_CA_2018']  # subset for fast demo

In [ ]:
# LandTrendr assets: reuse IDs from LandTrendr_AlphaEarth.ipynb
from src.change_detection import find_lt_band

assets = {
    'Angelina_Forest_TX': 'users/xihanyao/LT_Angelina_Forest_TX_31_95_NDVI_2016_2024',
    'Austin_Urban_TX': 'users/xihanyao/LT_Austin_UrbanGrowth_30_98_NDVI_2016_2024',
    'Bend_Urban_OR': 'users/xihanyao/LT_Bend_UrbanExpansion_44_121_NDVI_2016_2024',
    'Bootleg_Fire_OR_2021': 'users/xihanyao/LT_Bootleg_Fire_2021_43_121_NDVI_2016_2024',
    'Camp_Fire_CA_2018': 'users/xihanyao/LT_CampFire_CA_2018_40_122_NDVI_2016_2024',
    'CoosBay_Forest_OR': 'users/xihanyao/LT_CoosBay_IndustrialForestry_43_124_NDVI_2016_2024',
    'Dallas_Urban_TX': 'users/xihanyao/LT_Dallas_TX_33_97_NDVI_2016_2024',
    'Dixie_Fire_CA_2021': 'users/xihanyao/LT_DixieFire_CA_2021_40_121_NDVI_2016_2024',
    'Houston_Urban_TX': 'users/xihanyao/LT_Houston_TX_30_95_NDVI_2016_2024',
    'Mosquito_Fire_CA_2022': 'users/xihanyao/LT_MosquitoFire_CA_2022_39_121_NDVI_2016_2024',
    'MtHood_Forest_OR': 'users/xihanyao/LT_MtHood_WUI_Forestry_45_122_NDVI_2016_2024',
    'Portland_Urban_OR': 'users/xihanyao/LT_Portland_Metro_UrbanGrowth_45_123_NDVI_2016_2024',
    'Sacramento_Urban_CA': 'users/xihanyao/LT_Sacramento_UrbanEdge_39_121_NDVI_2016_2024',
    'Santiam_Fire_OR_2020': 'users/xihanyao/LT_Santiam_Fire_2020_45_122_NDVI_2016_2024',
    'ShastaTrinity_Forest_CA': 'users/xihanyao/LT_ShastaTrinity_Timberlands_41_122_NDVI_2016_2024',
}

images_lt = {k: ee.Image(v) for k, v in assets.items()}
print(f"Loaded {len(images_lt)} LandTrendr assets.")

# Build three combined LT layers across all sites
lt_yod_list = []
lt_mag_list = []
lt_dur_list = []

for name, img in images_lt.items():
    yod_b = find_lt_band(img, 'yod')
    mag_b = find_lt_band(img, 'mag')
    dur_b = find_lt_band(img, 'dur')
    geom = aois.get(name)
    if geom is None:
        geom = img.geometry()
    if yod_b:
        lt_yod_list.append(img.select(yod_b).clip(geom))
    if mag_b:
        lt_mag_list.append(img.select(mag_b).clip(geom))
    if dur_b:
        lt_dur_list.append(img.select(dur_b).clip(geom))

# Mosaic per band
lt_yod = ee.ImageCollection.fromImages(lt_yod_list).mosaic().rename('lt_yod') if lt_yod_list else None
lt_mag = ee.ImageCollection.fromImages(lt_mag_list).mosaic().rename('lt_mag') if lt_mag_list else None
lt_dur = ee.ImageCollection.fromImages(lt_dur_list).mosaic().rename('lt_dur') if lt_dur_list else None

print("LandTrendr combined layers prepared: ",
      f"YOD={lt_yod is not None}", f"MAG={lt_mag is not None}", f"DUR={lt_dur is not None}")

In [ ]:
# ── Site filter (respects RUN_HEAVY) ────────────────────────────────────────────
if not RUN_HEAVY:
    aois      = {k: v for k, v in aois.items()      if k in QUICK_SITES}
    assets    = {k: v for k, v in assets.items()    if k in QUICK_SITES}
    images_lt = {k: v for k, v in images_lt.items() if k in QUICK_SITES}
    # Rebuild LT mosaics for the filtered subset
    lt_yod = ee.ImageCollection.fromImages([img.select(
            next(b for b in img.bandNames().getInfo() if 'yod' in b.lower()))
        for img in images_lt.values()]).mosaic().rename('lt_yod')
    lt_mag = ee.ImageCollection.fromImages([img.select(
            next(b for b in img.bandNames().getInfo() if 'mag' in b.lower()))
        for img in images_lt.values()]).mosaic().rename('lt_mag')
    lt_dur = ee.ImageCollection.fromImages([img.select(
            next(b for b in img.bandNames().getInfo() if 'dur' in b.lower()))
        for img in images_lt.values()]).mosaic().rename('lt_dur')
    print(f"Quick mode: {len(aois)} sites → {list(aois.keys())}")
else:
    print(f"Full mode: {len(aois)} sites")


In [ ]:
# AlphaEarth change detection across all AOIs (2017–2024)
# Core logic is in src/change_detection.py — imported here instead of redefined
from src.change_detection import compute_alpha_layers

alpha_yod_list = []
alpha_mag_list = []
alpha_dur_list = []

for name, geom in aois.items():
    yod, mag, dur = compute_alpha_layers(geom, change_threshold=AE_MAG_THRESHOLD)
    alpha_yod_list.append(yod.clip(geom))
    alpha_mag_list.append(mag.clip(geom))
    alpha_dur_list.append(dur.clip(geom))

alpha_yod = ee.ImageCollection.fromImages(alpha_yod_list).mosaic() if alpha_yod_list else None
alpha_mag = ee.ImageCollection.fromImages(alpha_mag_list).mosaic() if alpha_mag_list else None
alpha_dur = ee.ImageCollection.fromImages(alpha_dur_list).mosaic() if alpha_dur_list else None

print("AlphaEarth combined layers prepared: ",
      f"YOD={alpha_yod is not None}", f"MAG={alpha_mag is not None}", f"DUR={alpha_dur is not None}")

In [ ]:
from ipywidgets import HTML
from ipyleaflet import Marker, DivIcon
from src.visualization import VIS_YOD, VIS_MAG_AE, VIS_MAG_LT, VIS_DUR

# Local aliases matching names used in addLayer calls below
vis_yod   = VIS_YOD
vis_mag_a = VIS_MAG_AE
vis_mag_l = VIS_MAG_LT
vis_dur_a = VIS_DUR
vis_dur_l = VIS_DUR

# Map visualization: combined layers for both methods
Map = geemap.Map(center=[37.5, -98], zoom=4)
Map.add_basemap('Esri.WorldImagery')

# Combined AOIs as one outline layer
fc_aois = ee.FeatureCollection([ee.Feature(geom, {'name': name}) for name, geom in aois.items()])
aoi_outline = ee.Image().byte().paint(fc_aois, 1, 2)
Map.addLayer(aoi_outline, {'palette': ['#00ffff']}, 'AOIs (all)', True)

Map.default_style = {'cursor': 'pointer'}
_last_marker = {'m': None}

def _on_map_click(**kwargs):
    if kwargs.get('type') != 'click':
        return
    lat, lon = kwargs.get('coordinates')
    pt = ee.Geometry.Point([lon, lat])
    feat = fc_aois.filterBounds(pt).first()
    name = None
    if feat:
        try:
            name = feat.get('name').getInfo()
        except Exception:
            name = None
    if _last_marker['m'] is not None:
        try:
            Map.remove(_last_marker['m'])
        except Exception:
            pass
        _last_marker['m'] = None
    text = f"<b>{name}</b>" if name else "<i>Outside AOIs</i>"
    marker = Marker(location=(lat, lon))
    marker.popup = HTML(value=text)
    Map.add(marker)
    _last_marker['m'] = marker

Map.on_interaction(_on_map_click)

# Add LandTrendr layers (combined)
if lt_yod:
    Map.addLayer(lt_yod, vis_yod, 'LandTrendr: YOD (all sites)', False)
if lt_mag:
    Map.addLayer(lt_mag, vis_mag_l, 'LandTrendr: MAG (all sites)', True)
if lt_dur:
    Map.addLayer(lt_dur, vis_dur_l, 'LandTrendr: DUR (all sites)', False)

# Add AlphaEarth layers (combined)
if alpha_yod:
    Map.addLayer(alpha_yod, vis_yod, 'AlphaEarth: YOC (all sites)', False)
if alpha_mag:
    Map.addLayer(alpha_mag, vis_mag_a, 'AlphaEarth: MAG (all sites)', True)
if alpha_dur:
    Map.addLayer(alpha_dur, vis_dur_a, 'AlphaEarth: DUR (all sites)', False)

Map

In [ ]:
# Apply smoothing and change-only masks for both methods
from src.change_detection import smooth_and_mask_alpha

# LandTrendr: smooth + threshold (MAG can be negative = vegetation loss)
if lt_mag is not None:
    gauss_kernel = ee.Kernel.gaussian(radius=AE_SMOOTH_RADIUS, sigma=AE_SMOOTH_SIGMA, units='pixels')
    lt_mag_smoothed = lt_mag.convolve(gauss_kernel)
    lt_change_mask  = lt_mag_smoothed.abs().gt(LT_MAG_THRESHOLD)
    lt_yod_masked   = lt_yod.updateMask(lt_change_mask) if lt_yod else None
    lt_mag_masked   = lt_mag_smoothed.updateMask(lt_change_mask)
    lt_dur_masked   = lt_dur.updateMask(lt_change_mask) if lt_dur else None
else:
    lt_mag_smoothed = lt_change_mask = lt_yod_masked = lt_mag_masked = lt_dur_masked = None

# AlphaEarth: smooth + threshold via shared utility
if alpha_mag is not None:
    alpha_mag_smoothed, alpha_change_mask, alpha_yod_masked, alpha_mag_masked, alpha_dur_masked = (
        smooth_and_mask_alpha(
            alpha_yod, alpha_mag, alpha_dur,
            mag_threshold=AE_MAG_THRESHOLD,
            radius=AE_SMOOTH_RADIUS,
            sigma=AE_SMOOTH_SIGMA,
        )
    )
else:
    alpha_mag_smoothed = alpha_change_mask = alpha_yod_masked = alpha_mag_masked = alpha_dur_masked = None

print("Applied smoothing and change-only masks:",
      f"LT mask={'ok' if lt_change_mask else 'none'};",
      f"Alpha mask={'ok' if alpha_change_mask else 'none'}")

In [ ]:
from ipywidgets import HTML
from ipyleaflet import Marker, DivIcon

# Vis params already imported in the cell above (vis_yod, vis_mag_a, etc.)

# Map visualization: masked layers for both methods
Map_Masked = geemap.Map(center=[37.5, -98], zoom=4)
Map_Masked.add_basemap('Esri.WorldImagery')

# Combined AOIs as one outline layer
fc_aois = ee.FeatureCollection([ee.Feature(geom, {'name': name}) for name, geom in aois.items()])
aoi_outline = ee.Image().byte().paint(fc_aois, 1, 2)
Map_Masked.addLayer(aoi_outline, {'palette': ['#00ffff']}, 'AOIs (all)', True)

Map_Masked.default_style = {'cursor': 'pointer'}
_last_marker_masked = {'m': None}

def _on_map_click_masked(**kwargs):
    if kwargs.get('type') != 'click':
        return
    lat, lon = kwargs.get('coordinates')
    pt = ee.Geometry.Point([lon, lat])
    feat = fc_aois.filterBounds(pt).first()
    name = None
    if feat:
        try:
            name = feat.get('name').getInfo()
        except Exception:
            name = None
    if _last_marker_masked['m'] is not None:
        try:
            Map_Masked.remove(_last_marker_masked['m'])
        except Exception:
            pass
        _last_marker_masked['m'] = None
    text = f"<b>{name}</b>" if name else "<i>Outside AOIs</i>"
    marker = Marker(location=(lat, lon))
    marker.popup = HTML(value=text)
    Map_Masked.add(marker)
    _last_marker_masked['m'] = marker

Map_Masked.on_interaction(_on_map_click_masked)

# Add LandTrendr layers (masked)
if lt_yod_masked:
    Map_Masked.addLayer(lt_yod_masked, vis_yod, 'LandTrendr: YOD (masked)', False)
if lt_mag_masked:
    Map_Masked.addLayer(lt_mag_masked, vis_mag_l, 'LandTrendr: MAG (smoothed+masked)', True)
if lt_dur_masked:
    Map_Masked.addLayer(lt_dur_masked, vis_dur_l, 'LandTrendr: DUR (masked)', False)

# Add AlphaEarth layers (masked)
if alpha_yod_masked:
    Map_Masked.addLayer(alpha_yod_masked, vis_yod, 'AlphaEarth: YOC (masked)', False)
if alpha_mag_masked:
    Map_Masked.addLayer(alpha_mag_masked, vis_mag_a, 'AlphaEarth: MAG (smoothed+masked)', True)
if alpha_dur_masked:
    Map_Masked.addLayer(alpha_dur_masked, vis_dur_a, 'AlphaEarth: DUR (masked)', False)

Map_Masked

In [ ]:
# IoU per AOI between LandTrendr and AlphaEarth change masks
import ee
import matplotlib.pyplot as plt

# Require masks
if "lt_change_mask" not in globals() or lt_change_mask is None:
    raise ValueError("lt_change_mask not found. Run the smoothing+mask cell first.")
if "alpha_change_mask" not in globals() or alpha_change_mask is None:
    raise ValueError("alpha_change_mask not found. Run the smoothing+mask cell first.")

scale   = 30    # meters (align with LandTrendr/Landsat)
max_px  = 1e13

# Pre-build intersection / union images once
inter_img = lt_change_mask.And(alpha_change_mask)
union_img  = lt_change_mask.Or(alpha_change_mask)
area_img   = ee.Image.pixelArea()           # band: 'area'

# ── Batched server-side computation ────────────────────────────────────────────
# Build a FeatureCollection so EE can parallelise all 15 sites in ONE round-trip.
fc_aois_ee = ee.FeatureCollection([
    ee.Feature(geom, {"name": name}) for name, geom in aois.items()
])

def _compute_iou(feature):
    geom = feature.geometry()
    inter_area = area_img.updateMask(inter_img).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=scale, maxPixels=max_px
    ).getNumber("area")
    union_area = area_img.updateMask(union_img).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=scale, maxPixels=max_px
    ).getNumber("area")
    iou = inter_area.divide(union_area.max(1e-10))
    return feature.set({"inter": inter_area, "union": union_area, "iou": iou})

result_fc   = fc_aois_ee.map(_compute_iou)

# Single getInfo() — all 15 sites fetched in one EE request
props = result_fc.aggregate_array("iou").getInfo()
names = result_fc.aggregate_array("name").getInfo()
inter_areas = result_fc.aggregate_array("inter").getInfo()
union_areas = result_fc.aggregate_array("union").getInfo()

# Replace Nones with 0 and compute final lists
ious  = [float(v) if v is not None else 0.0 for v in props]

# Sort by IoU descending
order        = sorted(range(len(names)), key=lambda i: ious[i], reverse=True)
names_sorted = [names[i]  for i in order]
ious_sorted  = [ious[i]   for i in order]

print("IoU results:")
for n, v in zip(names_sorted, ious_sorted):
    print(f"  {n:40s}  IoU = {v:.3f}")

plt.figure(figsize=(10, 6))
bars = plt.barh(names_sorted, ious_sorted, color="#4daf4a")
plt.gca().invert_yaxis()
plt.xlim(0, 1)
plt.xlabel("IoU (Intersection / Union)")
plt.title("IoU of Detected Change: LandTrendr vs AlphaEarth (by AOI)")
for b, v in zip(bars, ious_sorted):
    plt.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.2f}", va="center")
plt.tight_layout()
plt.savefig("../images/IoU_LandTrendr_AlphaEarth_15sites.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to images/IoU_LandTrendr_AlphaEarth_15sites.png")
